# Capítulo 1 — Leer los datos · Solución

*Matemáticas para la IA, paso a paso*, Belk Siri — notebook complementario.

[Abrir en Google Colab](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/es/ch01_corrige.ipynb) · Página complementaria: https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/es.html

Cada ejercicio retoma el enunciado del libro. La respuesta empieza con el razonamiento en papel y luego muestra su equivalente en Python.

**Convención.** Como en el libro, la varianza, la desviación típica, la puntuación z y la covarianza dividen por $n$ (`ddof=0`), salvo que se indique lo contrario.

**Índice.** Datos · Los cálculos de las lecciones · Ejercicios 1.1 a 1.6

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (7, 2.6)
# Colores del libro: un color = un papel (observado, predicho, desviación, recta del colega)
BLEU, ORANGE, ECART, VERT = '#2a78d6', '#eb6834', '#c2185b', '#2e8b57'

## Datos

Los identificadores (D0.1, D1.1…) son los del libro y de los archivos CSV.

Los nombres de variables y columnas del código se mantienen en francés: `logements` / `logement` = viviendas / vivienda, `surface_m2` = superficie en m², `prix_kEUR` = precio en k€, `nord` / `sud` = barrios Norte / Sur, `registre` = registro de ventas, `vente` = venta, `pieces` = habitaciones, `dpe` = clase energética, `delai_jours` = días hasta la venta, `etage` = planta, `delais_agences` = plazos de venta de las agencias, `tilleuls` / `port` = las dos residencias (Les Tilleuls, Le Port), `loyers` = alquileres, `moyenne` = media, `ecart(s)` = desviación(es).

In [ ]:
# Datos del capítulo 1, copiados de los archivos CSV del libro (no modificar)

# D0.1 — las seis viviendas de la apertura
logements = pd.DataFrame({
    'logement':   ['A', 'B', 'C', 'D', 'E', 'F'],
    'surface_m2': [32, 45, 54, 70, 85, 98],
    'prix_kEUR':  [118, 172, 185, 245, 279, 335],
})

# D1.1 — precios de venta de los T2 (dos habitaciones, k€), lección 1.1
nord = [174, 177, 181, 183, 185]
sud  = [150, 165, 185, 195, 205]

# D1.2 — registro de ventas del barrio Sur, lecciones 1.2 a 1.5
registre = pd.DataFrame({
    'vente':       ['S01', 'S02', 'S03', 'S04', 'S05', 'S06', 'S07', 'S08', 'S09', 'S10', 'S11', 'S12', 'S13'],
    'pieces':      [1, 1, 2, 2, 2, 2, 2, 3, 3, 3, 4, 4, 4],
    'surface_m2':  [24, 27, 43, 40, 44, 46, 52, 61, 64, 70, 82, 88, 95],
    'dpe':         ['D', 'C', 'F', 'D', 'C', 'D', 'C', 'E', 'D', 'B', 'E', 'C', 'G'],
    'prix_kEUR':   [96, 108, 150, 165, 185, 195, 205, 228, 241, 276, 289, 318, 260],
    'delai_jours': [9, 18, 97, 21, 26, 23, 31, 14, 35, 28, 42, 58, 183],
})

# D1.5 — precio por m² según la planta (k€/m²), lección 1.5
etages = pd.DataFrame({
    'etage':        [0, 1, 2, 3, 4, 5, 6],
    'prix_m2_kEUR': [3.0, 3.3, 3.5, 3.6, 3.5, 3.3, 3.0],
})

# D1.E1 — plazos de venta de tres agencias (días), ejercicio 1.1
delais_agences = {'P': [28, 29, 30, 31, 32], 'Q': [20, 25, 30, 35, 40], 'R': [10, 28, 30, 32, 50]}

# D1.E2 — alquileres mensuales de los estudios (€), ejercicio 1.2
tilleuls = [585, 590, 600, 605, 605, 615]
port     = [520, 560, 590, 620, 640, 670]

In [ ]:
# Herramientas proporcionadas: no tienes que modificarlas

def quartiles_livre(valeurs):
    """Q1 y Q3 según la regla del libro (ecuación 1.5): rangos ⌈n/4⌉ y ⌈3n/4⌉."""
    x = sorted(valeurs)
    n = len(x)
    rang_q1 = -(-n // 4)          # n/4 redondeado al entero superior
    rang_q3 = -(-3 * n // 4)      # 3n/4 redondeado al entero superior
    return x[rang_q1 - 1], x[rang_q3 - 1]   # rango 1 = posición 0 en Python


def points_par_rangee(series, xlabel, bandes=None, ax=None):
    """Una fila de puntos por serie, como en las figuras de la lección 1.1.
    bandes: diccionario {nombre: (media, desviación típica)} para dibujar media ± desviación típica."""
    if ax is None:
        fig, ax = plt.subplots()
    noms = list(series)
    couleurs = [BLEU, ORANGE, '#52514e']
    for i, nom in enumerate(noms):
        y = len(noms) - 1 - i
        if bandes and nom in bandes:
            m, s = bandes[nom]
            ax.add_patch(plt.Rectangle((m - s, y - 0.25), 2 * s, 0.5, color='#e9e8e4'))
            ax.plot([m, m], [y - 0.3, y + 0.3], color='black')
        ax.scatter(series[nom], [y] * len(series[nom]), s=50,
                   color=couleurs[i % 3], zorder=3)
    ax.set_yticks(range(len(noms)), list(reversed(noms)))
    ax.set_ylim(-0.6, len(noms) - 0.4)
    ax.set_xlabel(xlabel)
    ax.grid(axis='x', alpha=0.4)
    return ax

## Los cálculos de las lecciones, en Python

Estas celdas rehacen los cálculos de las lecciones 1.1 a 1.5. Ejecútalas y compáralas con el libro. Aquí, `np.std` y `np.var` se llaman con `ddof=0`, es decir, con una división por $n$, como en el capítulo.

In [ ]:
# Lección 1.1 — media, varianza y desviación típica (división por n)
for nom, serie in [('Norte', nord), ('Sur', sud)]:
    print(f'{nom}: media = {np.mean(serie):g} k€, varianza = {np.var(serie, ddof=0):g} k€², '
          f'desviación típica = {np.std(serie, ddof=0):g} k€')

In [ ]:
# Lección 1.2 — frecuencias por clase energética (variable cualitativa ordinal)
registre['dpe'].value_counts().reindex(list('ABCDEFG'), fill_value=0)

In [ ]:
# Lección 1.3 — media, mediana y cuartiles de los plazos
d = registre['delai_jours']
print('media:', d.mean(), ' mediana:', d.median())
print('sin S13: media', d[registre['vente'] != 'S13'].mean(),
      ' mediana', d[registre['vente'] != 'S13'].median())
print('cuartiles, regla del libro (ecuación 1.5):', quartiles_livre(d))
print('cuartiles, NumPy por defecto (interpolación):', np.quantile(d, [0.25, 0.75]))

In [ ]:
# Lección 1.4 — puntuaciones estandarizadas (ecuación 1.6, desviación típica con división por n)
for nom, serie, vente in [('Norte', nord, 188), ('Sur', sud, 210)]:
    m, s = np.mean(serie), np.std(serie, ddof=0)
    print(f'{nom}: z de la serie = {np.round((np.array(serie) - m) / s, 4)}, '
          f'z de la venta a {vente} k€ = {(vente - m) / s:g}')

In [ ]:
# Lección 1.5 — covarianza y correlación de los cinco T2 del Sur (división por n)
t2 = registre[registre['pieces'] == 2]
x, y = t2['surface_m2'].to_numpy(float), t2['prix_kEUR'].to_numpy(float)
cov = np.mean((x - x.mean()) * (y - y.mean()))
print('covarianza:', cov, 'm²·k€   r:', cov / (x.std(ddof=0) * y.std(ddof=0)))
print('r con np.corrcoef:', np.corrcoef(x, y)[0, 1].round(4))
print('r planta / precio por m²:', np.corrcoef(etages['etage'], etages['prix_m2_kEUR'])[0, 1].round(4))

## Ejercicio 1.1 — Leer un gráfico

*Datos D1.E1. Conceptos: media y desviación típica (lección 1.1).* El gráfico de abajo reproduce el del ejercicio 1.1 del libro.

In [ ]:
points_par_rangee({f'Agencia {a}': v for a, v in delais_agences.items()},
                  'Plazo de venta (días)')
plt.xlim(0, 60)
plt.show()

**a) y b)** Estima a ojo el plazo medio de cada agencia y luego ordena las agencias de la menos dispersa a la más dispersa.

In [ ]:
moyenne_estimee = {'P': 30, 'Q': 30, 'R': 30}   # en días
ordre_dispersion = ['P', 'Q', 'R']

In [ ]:
# Verificación (división por n)
for a, v in delais_agences.items():
    print(f'Agencia {a}: media = {np.mean(v):g} d, desviación típica = {np.std(v, ddof=0):.2f} d')
ordre_reel = sorted(delais_agences, key=lambda a: np.std(delais_agences[a]))
if ordre_dispersion is None or None in moyenne_estimee.values():
    print('Completa la celda anterior.')
else:
    print('Orden de dispersión:', 'correcto' if ordre_dispersion == ordre_reel else 'revisar')

**c)** Cuenta, para Q y para R, las ventas cerradas a 2 días o menos de la media.

In [ ]:
proches_Q = 1
proches_R = 3

In [ ]:
# Verificación
for a, la_votre in [('Q', proches_Q), ('R', proches_R)]:
    v = np.array(delais_agences[a])
    vrai = int(np.sum(np.abs(v - v.mean()) <= 2))
    if la_votre is None:
        print(f'Agencia {a}: aún sin completar.')
    else:
        print(f"Agencia {a}: {'correcto' if la_votre == vrai else 'revisar'}")

### Solución del ejercicio 1.1

**a)** Las tres series son simétricas en torno a 30 días. El plazo medio es de **30 días** para cada agencia, como confirma el cálculo: (28 + 29 + 30 + 31 + 32) / 5 = 150 / 5 = 30, e igual para Q y R.

**b)** De la menos a la más dispersa: **P, Q, R**. Los puntos de P caben entre 28 y 32 días. Los de Q se reparten regularmente de 20 a 40 días. Los de R van de 10 a 50 días. Desviaciones típicas (división por $n$): P ≈ 1,41 d; Q ≈ 7,07 d; R ≈ 12,71 d.

**c)** Q: **1** venta (30 días). R: **3** ventas (28, 30 y 32 días). Sin embargo, R tiene la mayor desviación típica. Sus dos ventas extremas, a 10 y 50 días, están a 20 días de la media. Sus cuadrados valen 400 cada uno: la suma de los cuadrados de R vale 400 + 4 + 0 + 4 + 400 = 808, frente a 100 + 25 + 0 + 25 + 100 = 250 para Q. La desviación típica resume todas las desviaciones, y los cuadrados dan mucho peso a las más grandes. Tres ventas cercanas al centro no compensan dos ventas muy alejadas.

*Elementos esperados:* el orden P, Q, R; los dos recuentos; el papel de los dos valores extremos y de los cuadrados en el cálculo.

## Ejercicio 1.2 — Dos residencias, un mismo alquiler medio

*Datos D1.E2. Conceptos: media, varianza y desviación típica (lección 1.1).* Camille no puede pagar más de 570 € al mes. Las preguntas a) a d) se hacen primero en papel, con la tabla del ejercicio 1.2 del libro. Este notebook sirve para verificar tus respuestas, después para programar la desviación típica (pregunta e) y comparar dos convenciones (pregunta f).

In [ ]:
points_par_rangee({'Les Tilleuls': tilleuls, 'Le Port': port}, "Alquiler mensual de un estudio (€)")
plt.axvline(570, color='black', linestyle=':', label='presupuesto de Camille: 570 €')
plt.xlim(500, 700)
plt.legend(loc='upper right', fontsize=8)
plt.show()

### a) Leer los alquileres

¿Cuántos estudios puede alquilar Camille en cada residencia con 570 €, y luego con 620 €?

In [ ]:
accessibles = {
    570: {'Les Tilleuls': 0, 'Le Port': 2},
    620: {'Les Tilleuls': 6, 'Le Port': 4},
}

In [ ]:
# Verificación: `<=` produce verdadero/falso, np.sum cuenta los «verdadero»
for budget, reponses in accessibles.items():
    for nom, serie in [('Les Tilleuls', tilleuls), ('Le Port', port)]:
        vrai = int(np.sum(np.array(serie) <= budget))
        la_votre = reponses[nom]
        etat = 'aún sin completar' if la_votre is None else ('correcto' if la_votre == vrai else 'revisar')
        print(f'Presupuesto {budget} €, {nom}: {etat}')

### Solución de 1.2 a)

**Leyendo la lista.** Con 570 €, Camille no encuentra **ningún** estudio en Les Tilleuls: el más barato cuesta 585 €. En Le Port, **dos** estudios de seis están a su alcance, a 520 y 560 €. La residencia donde tiene una oportunidad es, por tanto, la de alquileres más dispersos. Con 620 €, la situación se invierte: los **seis** estudios de Les Tilleuls están a su alcance, frente a **cuatro** de seis en Le Port (520, 560, 590 y 620 €). Las dos medias iguales no dejaban adivinar estas diferencias.

### b) Las dos medias

**En papel.** Les Tilleuls: 585 + 590 + 600 + 605 + 605 + 615 = 3600, y 3600 / 6 = **600 €**. Le Port: 520 + 560 + 590 + 620 + 640 + 670 = 3600, y 3600 / 6 = **600 €**.

**En Python.** `np.mean` hace la misma suma y luego la misma división.

In [ ]:
moyenne_tilleuls_main = 600
moyenne_port_main = 600

In [ ]:
# Verificación
for nom, serie, la_votre in [('Les Tilleuls', tilleuls, moyenne_tilleuls_main),
                              ('Le Port', port, moyenne_port_main)]:
    if la_votre is None:
        print(f'{nom}: completa la celda anterior.')
    else:
        ok = np.isclose(la_votre, np.mean(serie))
        print(f"{nom}: {'correcto' if ok else 'revisar'} (tu valor: {la_votre} €)")

### c) Varianza y desviación típica a mano (división por $n = 6$)

**Les Tilleuls** ($\bar{x} = 600$ €)

| $x_i$ (€) | 585 | 590 | 600 | 605 | 605 | 615 | Suma |
|:--|--:|--:|--:|--:|--:|--:|--:|
| $x_i - \bar{x}$ (€) | −15 | −10 | 0 | +5 | +5 | +15 | 0 |
| $(x_i - \bar{x})^2$ (€²) | 225 | 100 | 0 | 25 | 25 | 225 | 600 |

Varianza: 600 / 6 = **100 €²**. Desviación típica: √100 = **10 €**.

**Le Port** ($\bar{x} = 600$ €)

| $x_i$ (€) | 520 | 560 | 590 | 620 | 640 | 670 | Suma |
|:--|--:|--:|--:|--:|--:|--:|--:|
| $x_i - \bar{x}$ (€) | −80 | −40 | −10 | +20 | +40 | +70 | 0 |
| $(x_i - \bar{x})^2$ (€²) | 6400 | 1600 | 100 | 400 | 1600 | 4900 | 15 000 |

Varianza: 15 000 / 6 = **2500 €²**. Desviación típica: √2500 = **50 €**.

*Dos controles por el camino.* En cada tabla, la fila de las desviaciones debe sumar 0 (lección 1.1). Además, la desviación típica debe ser del mismo orden que las desviaciones de la tabla: 10 € para desviaciones de 0 a 15 €, 50 € para desviaciones de 10 a 80 €.

**En Python.** `np.var(..., ddof=0)` y `np.std(..., ddof=0)` dividen por $n$.

In [ ]:
variance_tilleuls_main = 100   # en €²
variance_port_main = 2500      # en €²
ecart_type_tilleuls_main = 10  # en €
ecart_type_port_main = 50      # en €

In [ ]:
# Verificación (división por n: ddof=0)
vos_valeurs = {
    'varianza Tilleuls': (variance_tilleuls_main, np.var(tilleuls, ddof=0)),
    'varianza Port': (variance_port_main, np.var(port, ddof=0)),
    'desviación típica Tilleuls': (ecart_type_tilleuls_main, np.std(tilleuls, ddof=0)),
    'desviación típica Port': (ecart_type_port_main, np.std(port, ddof=0)),
}
for nom, (la_votre, attendu) in vos_valeurs.items():
    if la_votre is None:
        print(f'{nom}: aún sin completar.')
    else:
        print(f"{nom}: {'correcto' if np.isclose(la_votre, attendu) else 'revisar'}")

### d) Interpretar

*Ejemplo de respuesta.* En Les Tilleuls, los alquileres suelen desviarse unos 10 € de 600 €: son casi todos iguales. En Le Port, la desviación habitual es de unos 50 €: hay estudios claramente más baratos y claramente más caros que la media. La desviación típica no basta para responder a la pregunta a). Describe la dispersión en general, pero no dice si un estudio concreto cuesta 570 € o menos. Una desviación típica de 50 € en torno a 600 € hace plausible un alquiler por debajo de 570 €. Solo la lista permite contar que hay exactamente dos, a 520 y 560 €.

*Elementos esperados:* una frase por residencia, con la unidad; la distinción entre una medida de dispersión y la pregunta «¿entra un estudio en el presupuesto?»; el recurso a la lista de alquileres.

### e) Programar la desviación típica

Completa la función siguiendo la ecuación 1.3 del libro, con una división por $n$:

$$\sigma = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (x_i - \bar{x})^2}$$

No uses ni `np.std` ni `np.var`: el objetivo es que escribas tú mismo cada paso.

In [ ]:
# Ejercicio 1.2 e)
def ecart_type(valeurs):
    """Desviación típica de una serie, con división por n (ecuación 1.3)."""
    x = np.array(valeurs, dtype=float)
    n = len(x)
    moyenne = np.sum(x) / n               # ecuación 1.1
    ecarts = x - moyenne                  # una resta por valor
    variance = np.sum(ecarts ** 2) / n    # ecuación 1.2
    return np.sqrt(variance)              # vuelta a euros

Cada línea de la función corresponde a una columna de la tabla de la pregunta c): las desviaciones, sus cuadrados, la media de esos cuadrados y, por último, la raíz cuadrada para volver a euros. Errores frecuentes: sacar la raíz antes de la media; calcular `np.sum(ecarts) / n`, que siempre da 0 (lección 1.1).

In [ ]:
# Verificación: ejecuta esta celda sin modificarla
if ecart_type(tilleuls) is None:
    print('La función todavía devuelve None: complétala.')
else:
    for nom, serie, attendu in [('Les Tilleuls', tilleuls, 10.0), ('Le Port', port, 50.0)]:
        obtenu = ecart_type(serie)
        etat = 'correcto' if np.isclose(obtenu, attendu) else f'revisar (esperado: {attendu})'
        print(f'{nom}: {obtenu:.2f} € → {etat}')

### f) Para profundizar: dividir por $n$ o por $n - 1$

Por defecto, el método `.std()` de pandas divide la suma de los cuadrados por $n - 1$ (`ddof=1`). NumPy divide por $n$ por defecto (`ddof=0`), la convención del capítulo.

**En papel, con división por $n - 1 = 5$.** Tilleuls: 600 / 5 = 120, y luego √120 ≈ **10,95 €**. Port: 15 000 / 5 = 3000, y luego √3000 ≈ **54,77 €**.

Las dos convenciones son correctas, pero no responden a la misma pregunta. Dividir por $n$ describe los seis estudios en sí mismos. Dividir por $n - 1$ sirve cuando estos seis estudios representan un conjunto más amplio. El capítulo 6 explicará esta elección. Con `.std(ddof=0)`, pandas vuelve a dar 10 y 50 €. En un cálculo, indica siempre la convención utilizada.

In [ ]:
loyers = pd.DataFrame({'Les Tilleuls': tilleuls, 'Le Port': port})
print('pandas, por defecto (división por n − 1):', loyers.std().round(2).to_dict())
print('pandas, ddof=0 (división por n)         :', loyers.std(ddof=0).round(2).to_dict())

## Ejercicio 1.3 — ¿Una venta sorprendente?

*Datos D1.1. Conceptos: puntuación estandarizada (lección 1.4, ecuación 1.6).* Un T2 (piso de dos habitaciones) acaba de venderse por 196 k€. Medias y desviaciones típicas: las de la lección 1.1, con división por $n$.

In [ ]:
z_nord = (196 - 180) / 4     # = 4
z_sud = (196 - 180) / 20     # = 0,8
prix_sud_aussi_exceptionnel = 180 + 4 * 20   # = 260 k€

In [ ]:
# Verificación (desviación típica con división por n)
attendus = {
    'z Norte': (z_nord, (196 - np.mean(nord)) / np.std(nord, ddof=0)),
    'z Sur': (z_sud, (196 - np.mean(sud)) / np.std(sud, ddof=0)),
    'precio Sur equivalente': (prix_sud_aussi_exceptionnel,
                            np.mean(sud) + (196 - np.mean(nord)) / np.std(nord, ddof=0) * np.std(sud, ddof=0)),
}
for nom, (la_votre, vrai) in attendus.items():
    etat = 'aún sin completar' if la_votre is None else ('correcto' if np.isclose(la_votre, vrai) else 'revisar')
    print(f'{nom}: {etat}')

### Solución del ejercicio 1.3

**a)** En el Norte: $z = (196 - 180)/4 = 16/4 =$ **4**. En el Sur: $z = (196 - 180)/20 = 16/20 =$ **0,8**. Las dos puntuaciones no tienen unidad: cuentan desviaciones típicas.

**b)** Sorprendería en el **Norte**. En la figura de las puntuaciones estandarizadas de la lección 1.4, una puntuación de 4 cae mucho más allá de todas las ventas del Norte, ninguna de las cuales supera 1,25. En el Sur, 0,8 se sitúa entre las ventas a 0,75 y 1,25: una venta corriente.

**c)** Buscamos el precio del Sur cuya puntuación vale 4: $180 + 4 \times 20 =$ **260 k€**. Hay que volver a la ecuación 1.6 y despejar $x$: $x = \bar{x} + z\,\sigma$.

*Elementos esperados:* las dos puntuaciones; la comparación en desviaciones típicas en lugar de en k€; el precio equivalente de 260 k€.

## Ejercicio 1.4 — ¿Van juntos superficie y precio?

*Datos D0.1 (las seis viviendas de la apertura). Conceptos: covarianza y correlación (lección 1.5, ecuaciones 1.7 y 1.8).*

In [ ]:
plt.figure(figsize=(5, 3.5))
plt.scatter(logements['surface_m2'], logements['prix_kEUR'], color=BLEU)
for _, l in logements.iterrows():
    plt.annotate(l['logement'], (l['surface_m2'], l['prix_kEUR']), xytext=(-10, 4), textcoords='offset points')
plt.xlabel('Superficie (m²)')
plt.ylabel('Precio de venta (k€)')
plt.grid(alpha=0.4)
plt.show()

**a)** Los puntos suben de izquierda a derecha y siguen casi una recta: $r$ será **positivo y muy cercano a 1**.

**b)** Completa las líneas de abajo. Usa una división por $n$ en todas partes.

In [ ]:
x = logements['surface_m2'].to_numpy(dtype=float)
y = logements['prix_kEUR'].to_numpy(dtype=float)
n = len(x)

produits = (x - x.mean()) * (y - y.mean())          # un producto por vivienda
covariance = np.sum(produits) / n                   # ecuación 1.7
r = covariance / (x.std(ddof=0) * y.std(ddof=0))    # ecuación 1.8

**En papel**, la superficie media vale 384 / 6 = 64 m² y el precio medio 1334 / 6 ≈ 222,33 k€. La suma de los seis productos vale 9825, de donde una covarianza de 9825 / 6 = **1637,5 m²·k€**. Las desviaciones típicas valen aproximadamente 22,80 m² y 72,15 k€, así que $r ≈ 1637{,}5 / (22{,}80 × 72{,}15) ≈$ **0,996**.

In [ ]:
# Verificación
if r is None or covariance is None:
    print('Completa la celda anterior.')
else:
    print('covarianza:', round(covariance, 2), 'm²·k€ →',
          'correcto' if np.isclose(covariance, 1637.5) else 'revisar (esperado: 1637,5)')
    print('r:', round(r, 4), '→', 'correcto' if np.isclose(r, np.corrcoef(x, y)[0, 1]) else 'revisar')

**Trampa que conviene conocer.** `np.cov(x, y)` divide por $n - 1$ por defecto y devuelve 1965, y no 1637,5. Para la convención del capítulo, hay que escribir `np.cov(x, y, bias=True)`. El coeficiente $r$, en cambio, no depende de la convención, porque el mismo factor aparece arriba y abajo (lección 1.5).

**c)** No hay contradicción. Primero, $r$ mide hasta qué punto los puntos se agrupan alrededor de *la* recta que mejor les conviene (la que el capítulo 3 te enseñará a calcular), no alrededor de nuestra recta trazada a ojo. Además, la cercanía es relativa: los precios se reparten con una desviación típica de unos 72 k€, y desviaciones de 5 a 11 k€ en valor absoluto son pequeñas al lado de esa dispersión. Un $r$ cercano a 1 no garantiza, por tanto, errores pequeños en k€. Para conocer el tamaño de los errores, hay que calcularlos, como en el ejercicio 1.6.

*Elementos esperados:* «$r$ se refiere a la mejor recta»; «las desviaciones son pequeñas en relación con la dispersión de los precios»; la distinción entre alineación y tamaño de los errores en k€.

In [ ]:
print('np.cov por defecto (división por n − 1):', np.cov(x, y)[0, 1])
print('np.cov con bias=True (división por n)  :', np.cov(x, y, bias=True)[0, 1])

## Ejercicio 1.5 — Predecir y luego verificar

*Datos D1.1. Conceptos: media y desviación típica (lección 1.1), mediana y rango intercuartílico (lección 1.3).* Un sexto T2 se vende en el Norte por 240 k€.

**a)** Predicción esperada: la media aumenta claramente, porque reparte los 240 k€ entre todas las ventas. La mediana apenas se mueve, porque la nueva venta solo se coloca «después del centro». La desviación típica aumenta mucho, porque una desviación de unos 50 k€ elevada al cuadrado domina la suma. El rango intercuartílico se mueve poco, porque solo depende de los valores centrales.

In [ ]:
nord_plus = nord + [240]
print('Norte:', sorted(nord))
print('Norte con la venta a 240 k€:', sorted(nord_plus))

**b)** Completa la función `resume`. Debe devolver los cuatro indicadores, con la desviación típica dividida por $n$ y los cuartiles calculados con la regla del libro (`quartiles_livre`, proporcionada más arriba). Las claves del diccionario se quedan en francés: `moyenne` = media, `médiane` = mediana, `écart-type` = desviación típica, `écart interquartile` = rango intercuartílico.

In [ ]:
def resume(serie):
    q1, q3 = quartiles_livre(serie)
    return {
        'moyenne': np.mean(serie),
        'médiane': np.median(serie),
        'écart-type': np.std(serie, ddof=0),   # división por n
        'écart interquartile': q3 - q1,
    }

In [ ]:
tableau = pd.DataFrame({'Norte (5 ventas)': resume(nord), 'Norte+240 k€ (6 ventas)': resume(nord_plus),
                        'Sur (5 ventas)': resume(sud)})
if tableau.isna().any().any():
    print('Completa la función resume.')
else:
    attendu = {'moyenne': (180, 190), 'médiane': (181, 182), 'écart-type': (4, 22.66),
               'écart interquartile': (6, 8)}
    for k, (avant, apres) in attendu.items():
        ok = np.isclose(tableau.loc[k].iloc[0], avant) and np.isclose(tableau.loc[k].iloc[1], apres, atol=0.005)
        print(f"{k}: {'correcto' if ok else 'revisar'}")
tableau.round(2)

In [ ]:
# Las dos series según el modelo de la figura «media ± desviación típica» de la lección 1.1
if tableau.isna().any().any():
    print('Completa la función resume para mostrar el gráfico.')
else:
    points_par_rangee({'Norte': nord, 'Norte + 240': nord_plus},
                      'Precio de venta de un T2 (k€)',
                      bandes={'Norte': (np.mean(nord), np.std(nord, ddof=0)),
                              'Norte + 240': (np.mean(nord_plus), np.std(nord_plus, ddof=0))})
    plt.xlim(140, 250)
    plt.show()

### Solución del ejercicio 1.5

**b) En papel.** Con la venta a 240 k€, la suma pasa de 900 a 1140 k€ para 6 ventas: media **190 k€** (en lugar de 180). Ordenados, los seis valores son 174, 177, 181, 183, 185, 240. $n$ es par, así que la mediana vale (181 + 183)/2 = **182 k€** (en lugar de 181). Las desviaciones respecto a 190 son −16, −13, −9, −7, −5, +50. Sus cuadrados valen 256, 169, 81, 49, 25 y 2500, para una suma de 3080. La varianza vale 3080/6 ≈ 513,3 k€², y la desviación típica **≈ 22,66 k€** (en lugar de 4). Cuartiles (ecuación 1.5): antes, rangos ⌈5/4⌉ = 2 y ⌈15/4⌉ = 4, es decir, Q1 = 177 y Q3 = 183, de donde un rango intercuartílico de **6 k€**. Después, rangos ⌈6/4⌉ = 2 y ⌈18/4⌉ = 5, es decir, Q1 = 177 y Q3 = 185, de donde **8 k€**.

**c)** No. Cinco ventas de seis siguen a 6 k€ o menos de 180 k€. Una sola venta excepcional basta para que la desviación típica pase de 4 a 22,66 k€, por encima de los 20 k€ del Sur, porque su desviación de 50 k€ cuenta 2500 en una suma de 3080. Las medidas robustas cuentan otra historia: el rango intercuartílico del Norte pasa de 6 a 8 k€ y sigue muy por debajo del del Sur (30 k€). Para un T2 corriente del Norte, los precios siguen siendo muy previsibles. La desviación típica señala sobre todo la presencia de una venta fuera de lo normal, que hay que mirar aparte.

*Elementos esperados:* los cuatro valores antes/después; el contraste entre indicadores sensibles (media, desviación típica) y robustos (mediana, rango intercuartílico); una conclusión matizada.

## Ejercicio 1.6 — Desafío: dos rectas, un mismo error medio

*Datos D0.1. Conceptos: desviación con signo y valor absoluto (apertura, lección 1.1), desviación típica (lección 1.1).* Recta trazada a ojo: precio predicho = 3 × superficie + 30. Recta perezosa: precio predicho = 222 k€ para todas las viviendas.

In [ ]:
prix = logements['prix_kEUR'].to_numpy(dtype=float)
surface = logements['surface_m2'].to_numpy(dtype=float)

ecarts_oeil = prix - (3 * surface + 30)
ecarts_paresseuse = prix - 222

In [ ]:
# Los tres indicadores de la pregunta b) (desviación típica: división por n)
if ecarts_oeil is None or ecarts_paresseuse is None:
    print('Completa la celda anterior.')
else:
    indicateurs = pd.DataFrame({
        nom: {'media de las desviaciones con signo': e.mean(),
              'media de las desviaciones en valor absoluto': np.abs(e).mean(),
              'desviación típica de las desviaciones con signo': e.std(ddof=0)}
        for nom, e in [('recta a ojo', ecarts_oeil), ('recta perezosa', ecarts_paresseuse)]
    })
    print('desviaciones, recta a ojo    :', ecarts_oeil)
    print('desviaciones, recta perezosa :', ecarts_paresseuse)
    print('verificación:', 'correcto' if np.allclose(indicateurs.iloc[:, 0], [1/3, 44/6, 7.5645], atol=1e-3)
          and np.allclose(indicateurs.iloc[:, 1], [1/3, 64, 72.1472], atol=1e-3) else 'revisar')
    display(indicateurs.round(2))

In [ ]:
if ecarts_oeil is not None and ecarts_paresseuse is not None:
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.axhline(0, color='black', linewidth=1)
    ax.scatter(surface - 0.8, ecarts_oeil, color=ORANGE, label="recta a ojo")
    ax.scatter(surface + 0.8, ecarts_paresseuse, color=BLEU, marker='s', label='recta perezosa')
    ax.set_xlabel('Superficie (m²)')
    ax.set_ylabel('Desviación observado − predicho (k€)')
    ax.legend()
    ax.grid(alpha=0.4)
    plt.show()

### Solución del ejercicio 1.6

**a)** Desviaciones de la recta perezosa (observado − 222): −104, −50, −37, +23, +57, +113 k€. Su suma vale 2 y su media ≈ **0,33 k€**. Es exactamente la media de las desviaciones con signo de la recta trazada a ojo (−8, +7, −7, +5, −6, +11: suma 2, media ≈ 0,33 k€).

**b)**

| | Recta a ojo | Recta perezosa |
|:--|--:|--:|
| Media de las desviaciones con signo | 0,33 k€ | 0,33 k€ |
| Media de las desviaciones en valor absoluto | 44/6 ≈ 7,33 k€ | 384/6 = 64 k€ |
| Desviación típica de las desviaciones con signo (división por $n$) | ≈ 7,56 k€ | ≈ 72,15 k€ |

Detalle para la recta a ojo: la media vale 1/3. Las desviaciones respecto a esta media tienen cuadrados cuya suma vale 344 − 6 × (1/3)² ≈ 343,33. La varianza vale 343,33/6 ≈ 57,22 y la desviación típica ≈ 7,56 k€.

**c)** La recta trazada a ojo es mucho mejor: sus errores son de unos 7 k€ en valor absoluto, frente a 64 k€ para la recta perezosa. La media de las desviaciones en valor absoluto y la desviación típica de las desviaciones con signo las distinguen. La media de las desviaciones con signo no lo consigue: ninguna de las dos rectas se inclina hacia un lado, y los signos se compensan en ambos casos.

**d)** Las desviaciones de la recta perezosa valen $y_i - 222$. Restar una misma constante desplaza toda la serie sin cambiar la distancia de cada valor a la media. La media también disminuye en 222, así que cada desviación respecto a la media $(y_i - 222) - (\bar{y} - 222) = y_i - \bar{y}$ queda idéntica. La desviación típica de las desviaciones es, por tanto, la de los precios, ≈ 72,15 k€. Una recta que ignora la superficie se equivoca, en promedio, en la propia dispersión de los precios. El capítulo 3 sacará partido de esta comparación.

*Elementos esperados:* las dos medias iguales; dos indicadores de tamaño del error claramente diferentes; la explicación por la compensación de los signos; el argumento del desplazamiento para d).

In [ ]:
# Control de d): la desviación típica de las desviaciones de la recta perezosa es igual a la de los precios
if ecarts_paresseuse is not None:
    print(round(ecarts_paresseuse.std(ddof=0), 4), round(prix.std(ddof=0), 4))